# 01 · Regresi Pasang Surut — Bab 2

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 2: membangun model regresi pertama (perceptron/MLP) untuk memprediksi tinggi pasang surut sederhana. Prasyarat: Bab 1 (`ch-01-00_fondasi_tensorflow`).

Versi TensorFlow dicetak di bawah agar hasil dapat direproduksi.

## 1. Setup & Verifikasi Lingkungan

In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0


GPU tersedia: []


## 2. Data Pasang Surut Sederhana (sintetik untuk latihan)

Contoh minimal: gelombang semi-diurnal dengan sedikit noise. Data nyata dibahas di Bab 8.

Fitur: tinggi air dua hari terakhir → target: tinggi air hari ini.

In [2]:
import numpy as np
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

t = np.arange(0, 500)
amp, period = 1.0, 12.42  # semi-diurnal ~12.42 jam

tinggi = amp * np.sin(2 * np.pi * t / period) + 0.05 * np.random.randn(len(t))

# Fitur: [tinggi(t-2), tinggi(t-1)], target: tinggi(t)
X = np.column_stack([tinggi[:-2], tinggi[1:-1]])
y = tinggi[2:]

print("X shape:", X.shape, "| y shape:", y.shape)

X shape: (498, 2) | y shape: (498,)


## 3. Split Berbasis Waktu

Untuk data deret waktu, jangan bagi acak. Melatih dengan data lama, menguji dengan data baru.

In [3]:
n = len(X)
n_train = int(n * 0.7)
n_val = int(n * 0.15)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]

print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape}")

train (348, 2) | val (74, 2) | test (76, 2)


## 4. Baseline: Persistence

Sebelum neural network, selalu ukur baseline sederhana (Bab 7 akan membahas lebih dalam):
prediksi = nilai kemarin.

In [4]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

base_pred = X_test[:, 1]  # persistence: pakai nilai t-1 (kolom fitur ke-2)
print("Baseline persistence MAE:", round(mae(y_test, base_pred), 4))

Baseline persistence MAE: 0.3256


## 5. Model MLP (Neural Network) Regresi

Perceptron/lapisan Dense + ReLU, lalu lapisan keluaran 1 neuron tanpa aktivasi (regresi).

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1),
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

C:\Users\Hi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105 (420.00 B)

 Trainable params: 105 (420.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    verbose=0,
)
print("Epoch terakhir - loss:", round(history.history["loss"][-1], 5),
      "| val_loss:", round(history.history["val_loss"][-1], 5))

Epoch terakhir - loss: 0.01067 | val_loss: 0.00888


In [7]:
pred = model.predict(X_test, verbose=0).ravel()
print("Model MLP MAE (test):", round(mae(y_test, pred), 4))
print("Baseline persistence MAE:", round(mae(y_test, base_pred), 4))
print("\n> Neural network layak jika mengalahkan baseline ini.")

Model MLP MAE (test): 0.0777
Baseline persistence MAE: 0.3256

> Neural network layak jika mengalahkan baseline ini.


## 6. Latihan Mini

1. Ganti jumlah unit `Dense` (mis. 32) — apa efeknya pada MAE?
2. Tambah fitur: `tinggi(t-3)` — apakah membantu?
3. Ubah loss `mse` → `mae` — bandingkan hasilnya.
4. Bandingkan MAE model vs persistence untuk horizon 1 hari.

Catat bahwa data di sini sintetik; data nyata pasang surut (dengan gap & noise) dibahas di Bab 6 dan Bab 8.